# Multi-Agent Social Learning on Kaggle GPU

**Project**: Accelerating Independent Agent Learning through Behavioral Estimation

This notebook implements the complete multi-agent social learning system to run on Kaggle's GPU.

## Setup Instructions
1. Upload this notebook to Kaggle
2. Enable GPU: Settings → Accelerator → GPU T4 x2
3. Run all cells in order

---

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q gymnasium matplotlib seaborn scipy tqdm

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

## 2. Environment Implementation

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class Navigation2DEnv(gym.Env):
    """2D Continuous Navigation Environment"""
    
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(self, dt=0.1, v_max=2.0, world_size=10.0, 
                 goal_threshold=0.5, max_steps=200, random_goal=False, render_mode=None):
        super().__init__()
        self.dt = dt
        self.v_max = v_max
        self.world_size = world_size
        self.goal_threshold = goal_threshold
        self.max_steps = max_steps
        self.random_goal = random_goal
        self.render_mode = render_mode

        # State space: [x, y, vx, vy, gx, gy]
        self.observation_space = spaces.Box(
            low=np.array([-world_size, -world_size, -v_max, -v_max, -world_size, -world_size]),
            high=np.array([world_size, world_size, v_max, v_max, world_size, world_size]),
            dtype=np.float32
        )

        # Action space: [ax, ay]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)

        self.state = None
        self.steps = 0
        self.goal = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        x = self.np_random.uniform(-self.world_size * 0.8, self.world_size * 0.8)
        y = self.np_random.uniform(-self.world_size * 0.8, self.world_size * 0.8)
        vx = self.np_random.uniform(-self.v_max * 0.5, self.v_max * 0.5)
        vy = self.np_random.uniform(-self.v_max * 0.5, self.v_max * 0.5)

        if self.random_goal or self.goal is None:
            gx = self.np_random.uniform(-self.world_size * 0.7, self.world_size * 0.7)
            gy = self.np_random.uniform(-self.world_size * 0.7, self.world_size * 0.7)
            self.goal = np.array([gx, gy], dtype=np.float32)

        gx, gy = self.goal
        self.state = np.array([x, y, vx, vy, gx, gy], dtype=np.float32)
        self.steps = 0
        return self.state, {}

    def step(self, action):
        action = np.clip(action, -1.0, 1.0)
        x, y, vx, vy, gx, gy = self.state
        ax, ay = action

        # Update velocity and position
        vx_new = np.clip(vx + self.dt * ax, -self.v_max, self.v_max)
        vy_new = np.clip(vy + self.dt * ay, -self.v_max, self.v_max)
        x_new = np.clip(x + self.dt * vx_new, -self.world_size, self.world_size)
        y_new = np.clip(y + self.dt * vy_new, -self.world_size, self.world_size)

        self.state = np.array([x_new, y_new, vx_new, vy_new, gx, gy], dtype=np.float32)
        self.steps += 1

        distance = np.sqrt((x_new - gx)**2 + (y_new - gy)**2)
        reward = -distance
        terminated = distance < self.goal_threshold
        truncated = self.steps >= self.max_steps

        return self.state, reward, terminated, truncated, {"distance": distance}

    def get_partial_observation(self, obs_type="full"):
        if obs_type == "position_only":
            return np.array([self.state[0], self.state[1], self.state[4], self.state[5]], dtype=np.float32)
        else:
            return self.state.copy()

    def render(self):
        pass  # Skip rendering in Kaggle

    def close(self):
        pass

print("✓ Environment created")

## 3. Neural Network Architectures

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

LOG_STD_MIN = -20
LOG_STD_MAX = 2

def weights_init_(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight, gain=1)
        torch.nn.init.constant_(m.bias, 0)

class QNetwork(nn.Module):
    """Critic Network"""
    def __init__(self, obs_dim, action_dim, hidden_dim=256, social_embed_dim=0):
        super().__init__()
        input_dim = obs_dim + action_dim + social_embed_dim
        self.social_embed_dim = social_embed_dim
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.apply(weights_init_)

    def forward(self, obs, action, social_embed=None):
        if social_embed is not None:
            x = torch.cat([obs, action, social_embed], dim=-1)
        else:
            if self.social_embed_dim > 0:
                batch_size = obs.shape[0]
                zero_embed = torch.zeros(batch_size, self.social_embed_dim, device=obs.device)
                x = torch.cat([obs, action, zero_embed], dim=-1)
            else:
                x = torch.cat([obs, action], dim=-1)
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class GaussianPolicy(nn.Module):
    """Actor Network"""
    def __init__(self, obs_dim, action_dim, hidden_dim=256, social_embed_dim=0):
        super().__init__()
        input_dim = obs_dim + social_embed_dim
        self.social_embed_dim = social_embed_dim
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)
        self.apply(weights_init_)

    def forward(self, obs, social_embed=None):
        if social_embed is not None:
            x = torch.cat([obs, social_embed], dim=-1)
        else:
            if self.social_embed_dim > 0:
                batch_size = obs.shape[0]
                zero_embed = torch.zeros(batch_size, self.social_embed_dim, device=obs.device)
                x = torch.cat([obs, zero_embed], dim=-1)
            else:
                x = obs
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        mean = self.mean(x)
        log_std = torch.clamp(self.log_std(x), LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    def sample(self, obs, social_embed=None):
        mean, log_std = self.forward(obs, social_embed)
        std = log_std.exp()
        normal = Normal(mean, std)
        x_t = normal.rsample()
        action = torch.tanh(x_t)
        log_prob = normal.log_prob(x_t) - torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(-1, keepdim=True)
        mean = torch.tanh(mean)
        return action, log_prob, mean

class SocialEmbeddingNetwork(nn.Module):
    """Social Observation Encoder"""
    def __init__(self, state_dim, action_dim, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.embed_dim = embed_dim
        
        per_agent_input = state_dim + action_dim
        self.encoder = nn.Sequential(
            nn.Linear(per_agent_input, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU()
        )
        self.aggregator = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim), nn.ReLU()
        )
        self.apply(weights_init_)

    def forward(self, social_obs):
        batch_size, n_agents, _ = social_obs.shape
        social_flat = social_obs.reshape(-1, self.state_dim + self.action_dim)
        encoded = self.encoder(social_flat).reshape(batch_size, n_agents, -1)
        aggregated = encoded.mean(dim=1)
        return self.aggregator(aggregated)

class ActionPredictionHead(nn.Module):
    """Auxiliary Action Predictor"""
    def __init__(self, embed_dim, action_dim):
        super().__init__()
        self.predictor = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(),
            nn.Linear(128, action_dim), nn.Tanh()
        )
        self.apply(weights_init_)

    def forward(self, embedding):
        return self.predictor(embedding)

print("✓ Networks created")

## 4. SAC Agent Implementation

In [ ]:
import torch.optim as optim

class ReplayBuffer:
    def __init__(self, capacity, obs_dim, action_dim, device):
        self.capacity = capacity
        self.device = device
        self.ptr = 0
        self.size = 0
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.action = np.zeros((capacity, action_dim), dtype=np.float32)
        self.reward = np.zeros((capacity, 1), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.done = np.zeros((capacity, 1), dtype=np.float32)

    def add(self, obs, action, reward, next_obs, done):
        self.obs[self.ptr] = obs
        self.action[self.ptr] = action
        self.reward[self.ptr] = reward
        self.next_obs[self.ptr] = next_obs
        self.done[self.ptr] = done
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self.size, size=batch_size)
        return dict(
            obs=torch.FloatTensor(self.obs[idx]).to(self.device),
            action=torch.FloatTensor(self.action[idx]).to(self.device),
            reward=torch.FloatTensor(self.reward[idx]).to(self.device),
            next_obs=torch.FloatTensor(self.next_obs[idx]).to(self.device),
            done=torch.FloatTensor(self.done[idx]).to(self.device)
        )

class SocialReplayBuffer(ReplayBuffer):
    def __init__(self, capacity, obs_dim, action_dim, social_obs_dim, device):
        super().__init__(capacity, obs_dim, action_dim, device)
        self.social_obs = np.zeros((capacity, social_obs_dim), dtype=np.float32)
        self.next_social_obs = np.zeros((capacity, social_obs_dim), dtype=np.float32)

    def add(self, obs, action, reward, next_obs, done, social_obs=None, next_social_obs=None):
        super().add(obs, action, reward, next_obs, done)
        if social_obs is not None:
            self.social_obs[self.ptr - 1] = social_obs
        if next_social_obs is not None:
            self.next_social_obs[self.ptr - 1] = next_social_obs

    def sample(self, batch_size):
        batch = super().sample(batch_size)
        idx = np.random.randint(0, self.size, size=batch_size)
        batch['social_obs'] = torch.FloatTensor(self.social_obs[idx]).to(self.device)
        batch['next_social_obs'] = torch.FloatTensor(self.next_social_obs[idx]).to(self.device)
        return batch

print("✓ Replay buffers created")

In [ ]:
# SAC Agent class (continues from previous cell)

class SACAgent:
    def __init__(self, obs_dim, action_dim, device, method="independent",
                 social_state_dim=0, social_action_dim=0, n_other_agents=0,
                 lr=3e-4, gamma=0.99, tau=0.005, alpha=0.2,
                 auto_entropy_tuning=True, hidden_dim=256,
                 social_embed_dim=64, aux_loss_weight=0.1):
        
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.device = device
        self.method = method
        self.gamma = gamma
        self.tau = tau
        self.alpha = alpha
        self.auto_entropy_tuning = auto_entropy_tuning
        self.aux_loss_weight = aux_loss_weight
        self.social_state_dim = social_state_dim
        self.social_action_dim = social_action_dim
        self.n_other_agents = n_other_agents

        policy_input_dim = obs_dim
        social_embedding_dim = 0

        if method == "concat":
            policy_input_dim += n_other_agents * (social_state_dim + social_action_dim)
        elif method == "proposed":
            social_embedding_dim = social_embed_dim
            self.social_encoder = SocialEmbeddingNetwork(
                social_state_dim, social_action_dim, social_embed_dim
            ).to(device)
            self.social_encoder_optimizer = optim.Adam(self.social_encoder.parameters(), lr=lr)
            self.action_predictor = ActionPredictionHead(social_embed_dim, social_action_dim).to(device)
            self.action_predictor_optimizer = optim.Adam(self.action_predictor.parameters(), lr=lr)

        self.policy = GaussianPolicy(policy_input_dim, action_dim, hidden_dim, social_embedding_dim).to(device)
        self.critic_1 = QNetwork(policy_input_dim, action_dim, hidden_dim, social_embedding_dim).to(device)
        self.critic_2 = QNetwork(policy_input_dim, action_dim, hidden_dim, social_embedding_dim).to(device)
        self.critic_target_1 = QNetwork(policy_input_dim, action_dim, hidden_dim, social_embedding_dim).to(device)
        self.critic_target_2 = QNetwork(policy_input_dim, action_dim, hidden_dim, social_embedding_dim).to(device)

        self.critic_target_1.load_state_dict(self.critic_1.state_dict())
        self.critic_target_2.load_state_dict(self.critic_2.state_dict())

        self.policy_optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.critic_1_optimizer = optim.Adam(self.critic_1.parameters(), lr=lr)
        self.critic_2_optimizer = optim.Adam(self.critic_2.parameters(), lr=lr)

        if self.auto_entropy_tuning:
            self.target_entropy = -action_dim
            self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
            self.alpha_optimizer = optim.Adam([self.log_alpha], lr=lr)

    def select_action(self, obs, social_obs=None, deterministic=False):
        obs = torch.FloatTensor(obs).unsqueeze(0).to(self.device)
        social_embed = None
        
        if self.method == "concat":
            if social_obs is not None:
                social_flat = social_obs.flatten()
            else:
                social_size = self.policy.fc1.in_features - self.obs_dim
                social_flat = np.zeros(social_size, dtype=np.float32)
            obs = torch.cat([obs, torch.FloatTensor(social_flat).unsqueeze(0).to(self.device)], dim=-1)
        elif self.method == "proposed":
            if social_obs is not None:
                social_tensor = torch.FloatTensor(social_obs).unsqueeze(0).to(self.device)
                social_embed = self.social_encoder(social_tensor)

        with torch.no_grad():
            if deterministic:
                _, _, action = self.policy.sample(obs, social_embed)
            else:
                action, _, _ = self.policy.sample(obs, social_embed)
        return action.cpu().numpy()[0]

    def update(self, batch, step):
        obs, action, reward = batch['obs'], batch['action'], batch['reward']
        next_obs, done = batch['next_obs'], batch['done']
        
        social_embed = None
        next_social_embed = None
        aux_loss = torch.tensor(0.0).to(self.device)

        if self.method == "proposed" and 'social_obs' in batch:
            social_obs = batch['social_obs']
            next_social_obs = batch['next_social_obs']
            batch_size = obs.shape[0]
            features_per_agent = self.social_state_dim + self.social_action_dim
            social_obs = social_obs.reshape(batch_size, self.n_other_agents, features_per_agent)
            next_social_obs = next_social_obs.reshape(batch_size, self.n_other_agents, features_per_agent)
            social_embed = self.social_encoder(social_obs)
            next_social_embed = self.social_encoder(next_social_obs)
            predicted_actions = self.action_predictor(social_embed)
            actual_actions = social_obs[:, :, self.social_state_dim:]
            actual_actions_mean = actual_actions.mean(dim=1)
            aux_loss = F.mse_loss(predicted_actions, actual_actions_mean)
        elif self.method == "concat" and 'social_obs' in batch:
            obs = torch.cat([obs, batch['social_obs']], dim=-1)
            next_obs = torch.cat([next_obs, batch['next_social_obs']], dim=-1)

        # Update critics
        with torch.no_grad():
            next_action, next_log_prob, _ = self.policy.sample(next_obs, next_social_embed)
            target_q1 = self.critic_target_1(next_obs, next_action, next_social_embed)
            target_q2 = self.critic_target_2(next_obs, next_action, next_social_embed)
            target_q = torch.min(target_q1, target_q2) - self.alpha * next_log_prob
            target_q = reward + (1 - done) * self.gamma * target_q

        social_embed_detached = social_embed.detach() if social_embed is not None else None
        current_q1 = self.critic_1(obs, action, social_embed_detached)
        current_q2 = self.critic_2(obs, action, social_embed_detached)
        critic_1_loss = F.mse_loss(current_q1, target_q)
        critic_2_loss = F.mse_loss(current_q2, target_q)

        self.critic_1_optimizer.zero_grad()
        critic_1_loss.backward()
        self.critic_1_optimizer.step()

        self.critic_2_optimizer.zero_grad()
        critic_2_loss.backward()
        self.critic_2_optimizer.step()

        # Update policy
        new_action, log_prob, _ = self.policy.sample(obs, social_embed)
        q1 = self.critic_1(obs, new_action, social_embed)
        q2 = self.critic_2(obs, new_action, social_embed)
        q = torch.min(q1, q2)
        policy_loss = (self.alpha * log_prob - q).mean()

        if self.method == "proposed":
            total_loss = policy_loss + self.aux_loss_weight * aux_loss
        else:
            total_loss = policy_loss

        self.policy_optimizer.zero_grad()
        if self.method == "proposed":
            self.social_encoder_optimizer.zero_grad()
            self.action_predictor_optimizer.zero_grad()
        total_loss.backward()
        self.policy_optimizer.step()
        if self.method == "proposed":
            self.social_encoder_optimizer.step()
            self.action_predictor_optimizer.step()

        # Update alpha
        if self.auto_entropy_tuning:
            alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()
            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()
            self.alpha = self.log_alpha.exp()

        # Soft update targets
        for target_param, param in zip(self.critic_target_1.parameters(), self.critic_1.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        for target_param, param in zip(self.critic_target_2.parameters(), self.critic_2.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        return {
            'critic_1_loss': critic_1_loss.item(),
            'critic_2_loss': critic_2_loss.item(),
            'policy_loss': policy_loss.item(),
            'alpha': self.alpha.item() if isinstance(self.alpha, torch.Tensor) else self.alpha,
            'aux_loss': aux_loss.item() if isinstance(aux_loss, torch.Tensor) else 0.0
        }

print("✓ SAC Agent created")

## 5. Social Observation Channel

In [ ]:
class SocialObservationChannel:
    def __init__(self, n_agents, state_noise_std=0.1, action_noise_std=0.05):
        self.n_agents = n_agents
        self.state_noise_std = state_noise_std
        self.action_noise_std = action_noise_std
        self.states = [None] * n_agents
        self.actions = [None] * n_agents

    def update(self, agent_id, state, action):
        self.states[agent_id] = state
        self.actions[agent_id] = action

    def get_social_observation(self, agent_id, add_noise=True):
        social_obs = []
        for i in range(self.n_agents):
            if i == agent_id:
                continue
            if self.states[i] is None or self.actions[i] is None:
                sa_pair = np.zeros(8, dtype=np.float32)
            else:
                state = self.states[i].copy()
                action = self.actions[i].copy()
                if add_noise:
                    state += np.random.normal(0, self.state_noise_std, size=state.shape)
                    action += np.random.normal(0, self.action_noise_std, size=action.shape)
                    action = np.clip(action, -1.0, 1.0)
                sa_pair = np.concatenate([state, action])
            social_obs.append(sa_pair)
        return np.array(social_obs, dtype=np.float32)

    def reset(self):
        self.states = [None] * self.n_agents
        self.actions = [None] * self.n_agents

class AgentConfig:
    def __init__(self, agent_id, obs_type="full", expertise_level="novice", pretrain_steps=0):
        self.agent_id = agent_id
        self.obs_type = obs_type
        self.expertise_level = expertise_level
        self.pretrain_steps = pretrain_steps

    def get_obs_dim(self, env):
        return 4 if self.obs_type == "position_only" else 6

def create_default_agent_configs():
    return [
        AgentConfig(0, obs_type="position_only", expertise_level="novice", pretrain_steps=0),
        AgentConfig(1, obs_type="full", expertise_level="novice", pretrain_steps=0),
        AgentConfig(2, obs_type="full", expertise_level="expert", pretrain_steps=50000)
    ]

print("✓ Social channel and configs created")

## 6. Training Function

In [ ]:
from tqdm.notebook import tqdm

def train_multi_agent(method="proposed", total_steps=100000, n_agents=3, 
                     batch_size=256, device="cuda", seed=0):
    
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # Create environments and agents
    envs = [Navigation2DEnv(random_goal=True) for _ in range(n_agents)]
    agent_configs = create_default_agent_configs()
    social_channel = SocialObservationChannel(n_agents, 0.1, 0.05)
    
    agents = []
    buffers = []
    
    for i, config in enumerate(agent_configs):
        obs_dim = config.get_obs_dim(envs[i])
        action_dim = 2
        
        if method == "independent":
            social_state_dim, social_action_dim, n_other_agents = 0, 0, 0
        else:
            social_state_dim, social_action_dim, n_other_agents = 6, 2, n_agents - 1
        
        agent = SACAgent(obs_dim, action_dim, device, method, 
                        social_state_dim, social_action_dim, n_other_agents)
        agents.append(agent)
        
        if method == "independent":
            buffer = ReplayBuffer(1000000, obs_dim, action_dim, device)
        else:
            social_obs_dim = n_other_agents * (social_state_dim + social_action_dim)
            buffer = SocialReplayBuffer(1000000, obs_dim, action_dim, social_obs_dim, device)
        buffers.append(buffer)
    
    # Pretrain expert
    for i, config in enumerate(agent_configs):
        if config.pretrain_steps > 0:
            print(f"Pretraining Agent {i} for {config.pretrain_steps} steps...")
            env = envs[i]
            agent = agents[i]
            buffer = buffers[i]
            obs, _ = env.reset()
            obs = env.get_partial_observation(config.obs_type)
            
            for step in tqdm(range(config.pretrain_steps), desc=f"Pretrain Agent {i}"):
                action = env.action_space.sample() if step < 10000 else agent.select_action(obs)
                next_obs_full, reward, terminated, truncated, _ = env.step(action)
                next_obs = env.get_partial_observation(config.obs_type)
                done = terminated or truncated
                buffer.add(obs, action, reward, next_obs, float(done))
                obs = next_obs
                if step >= 10000 and step % 1 == 0:
                    agent.update(buffer.sample(batch_size), step)
                if done:
                    obs, _ = env.reset()
                    obs = env.get_partial_observation(config.obs_type)
    
    # Main training
    observations = [env.get_partial_observation(agent_configs[i].obs_type) 
                   for i, (env, _) in enumerate([(e, e.reset()) for e in envs])]
    social_channel.reset()
    
    episode_rewards = [[] for _ in range(n_agents)]
    total_rewards = [0.0] * n_agents
    
    pbar = tqdm(total=total_steps, desc=f"Training ({method})")
    for global_step in range(total_steps):
        for i in range(n_agents):
            env = envs[i]
            agent = agents[i]
            buffer = buffers[i]
            config = agent_configs[i]
            obs = observations[i]
            
            social_obs = None if method == "independent" else social_channel.get_social_observation(i)
            
            if global_step < 10000:
                action = env.action_space.sample()
            else:
                action = agent.select_action(obs, social_obs.flatten() if method == "concat" and social_obs is not None else social_obs)
            
            next_obs_full, reward, terminated, truncated, _ = env.step(action)
            next_obs = env.get_partial_observation(config.obs_type)
            done = terminated or truncated
            
            social_channel.update(i, next_obs_full, action)
            next_social_obs = None if method == "independent" else social_channel.get_social_observation(i)
            
            if method == "independent":
                buffer.add(obs, action, reward, next_obs, float(done))
            else:
                buffer.add(obs, action, reward, next_obs, float(done), 
                          social_obs.flatten(), next_social_obs.flatten())
            
            total_rewards[i] += reward
            observations[i] = next_obs
            
            if global_step >= 10000 and global_step % 1 == 0:
                agent.update(buffer.sample(batch_size), global_step)
            
            if done:
                episode_rewards[i].append(total_rewards[i])
                total_rewards[i] = 0.0
                obs, _ = env.reset()
                observations[i] = env.get_partial_observation(config.obs_type)
        
        pbar.update(1)
        if global_step % 10000 == 0 and global_step > 0:
            avg_rewards = [np.mean(er[-10:]) if len(er) > 0 else 0 for er in episode_rewards]
            pbar.set_postfix({f"A{i}": f"{avg_rewards[i]:.1f}" for i in range(n_agents)})
    
    pbar.close()
    return agents, episode_rewards

print("✓ Training function ready")

## 7. Run Training

In [ ]:
# Quick test training (adjust total_steps for full training)
print(f"\n{'='*60}")
print("Starting Training")
print(f"{'='*60}\n")

# Train with proposed method
# Change total_steps to 500000 for full training
agents, episode_rewards = train_multi_agent(
    method="proposed",
    total_steps=100000,  # Increase to 500000 for full training
    device=device,
    seed=0
)

print(f"\n{'='*60}")
print("Training Complete!")
print(f"{'='*60}\n")

## 8. Visualize Results

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
agent_names = ['Agent 0 (pos only)', 'Agent 1 (full, novice)', 'Agent 2 (full, expert)']

for i in range(3):
    if len(episode_rewards[i]) > 0:
        # Smooth rewards
        smoothed = []
        window = 10
        for j in range(len(episode_rewards[i])):
            start = max(0, j - window)
            smoothed.append(np.mean(episode_rewards[i][start:j+1]))
        
        axes[i].plot(smoothed, alpha=0.8, linewidth=2)
        axes[i].set_title(agent_names[i], fontsize=12, fontweight='bold')
        axes[i].set_xlabel('Episode')
        axes[i].set_ylabel('Return')
        axes[i].grid(True, alpha=0.3)
        
        # Show final performance
        final_avg = np.mean(episode_rewards[i][-10:]) if len(episode_rewards[i]) >= 10 else 0
        axes[i].axhline(y=final_avg, color='r', linestyle='--', alpha=0.5, 
                       label=f'Final avg: {final_avg:.1f}')
        axes[i].legend()

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFinal Performance:")
print("="*50)
for i in range(3):
    if len(episode_rewards[i]) >= 10:
        final_rewards = episode_rewards[i][-10:]
        print(f"{agent_names[i]}: {np.mean(final_rewards):.2f} ± {np.std(final_rewards):.2f}")
print("="*50)

## 9. Evaluate Trained Agents

In [ ]:
# Evaluate agents
def evaluate_agent(agent, agent_config, n_episodes=10):
    env = Navigation2DEnv(random_goal=True)
    eval_rewards = []
    
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        obs = env.get_partial_observation(agent_config.obs_type)
        episode_reward = 0
        done = False
        
        while not done:
            action = agent.select_action(obs, deterministic=True)
            next_obs_full, reward, terminated, truncated, _ = env.step(action)
            obs = env.get_partial_observation(agent_config.obs_type)
            episode_reward += reward
            done = terminated or truncated
        
        eval_rewards.append(episode_reward)
    
    env.close()
    return np.mean(eval_rewards), np.std(eval_rewards)

print("\nEvaluation Results (10 episodes each):")
print("="*60)
agent_configs = create_default_agent_configs()
for i, (agent, config) in enumerate(zip(agents, agent_configs)):
    mean_reward, std_reward = evaluate_agent(agent, config, n_episodes=10)
    print(f"Agent {i} ({config.obs_type}, {config.expertise_level}):")
    print(f"  Average Reward: {mean_reward:.2f} ± {std_reward:.2f}")
print("="*60)

## 10. Save Models (Optional)

In [ ]:
# Save trained models
Path("models").mkdir(exist_ok=True)

for i, agent in enumerate(agents):
    checkpoint = {
        'policy': agent.policy.state_dict(),
        'critic_1': agent.critic_1.state_dict(),
        'critic_2': agent.critic_2.state_dict(),
    }
    if hasattr(agent, 'social_encoder'):
        checkpoint['social_encoder'] = agent.social_encoder.state_dict()
        checkpoint['action_predictor'] = agent.action_predictor.state_dict()
    
    torch.save(checkpoint, f"models/agent_{i}.pt")
    print(f"✓ Saved Agent {i}")

# Save results
results = {
    'episode_rewards': [list(map(float, er)) for er in episode_rewards],
    'device': device,
    'method': 'proposed'
}
with open('models/results.json', 'w') as f:
    json.dump(results, f)

print("\n✓ All models and results saved!")

## 11. Comparison Across Methods (Optional - Run Multiple Times)

In [ ]:
# Train with all three methods for comparison
# This takes a while - only run if you want full comparison

comparison_results = {}

for method in ['independent', 'concat', 'proposed']:
    print(f"\nTraining with method: {method}")
    agents_m, rewards_m = train_multi_agent(
        method=method,
        total_steps=50000,  # Shorter for comparison
        device=device,
        seed=0
    )
    comparison_results[method] = rewards_m

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
colors = {'independent': 'blue', 'concat': 'orange', 'proposed': 'green'}

for agent_id in range(3):
    for method, rewards in comparison_results.items():
        if len(rewards[agent_id]) > 0:
            smoothed = []
            window = 10
            for j in range(len(rewards[agent_id])):
                start = max(0, j - window)
                smoothed.append(np.mean(rewards[agent_id][start:j+1]))
            axes[agent_id].plot(smoothed, label=method.capitalize(), 
                              color=colors[method], alpha=0.7, linewidth=2)
    
    axes[agent_id].set_title(agent_names[agent_id], fontsize=12, fontweight='bold')
    axes[agent_id].set_xlabel('Episode')
    axes[agent_id].set_ylabel('Return')
    axes[agent_id].legend()
    axes[agent_id].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary and Next Steps

### What You've Accomplished:
✅ Implemented complete multi-agent social learning system  
✅ Trained agents with GPU acceleration  
✅ Evaluated and visualized results  

### Expected Results:
- **Agent 2 (Expert)**: Best performance due to pretraining
- **Agent 1 (Full obs, Novice)**: Better than Agent 0 due to complete observations
- **Agent 0 (Partial obs, Novice)**: Limited by partial observations but benefits from social learning

### For Better Results:
1. **Increase training steps**: Set `total_steps=500000` for full training
2. **Multiple seeds**: Run with different seeds and average results
3. **Compare methods**: Train with all three methods (independent, concat, proposed)

### Download Results:
- Click on the output files in the left sidebar
- Download `learning_curves.png` and model files from `models/`
- Use these for your project report!

---

**Project**: Multi-Agent Social Learning  
**GPU**: Kaggle T4 x2  
**Status**: ✅ Complete
